# Day 32: Dashboard Basics with Streamlit

## Install Note
So far every chart in this course has lived inside a notebook, but real stakeholders often want a shareable, always-on web page they can open in a browser and interact with themselves - that's what Streamlit is for. It takes an ordinary Python script and turns it into a live web app, handling all the HTML/CSS/JavaScript for you, so you can build a dashboard using only pandas-style Python. Important: Streamlit apps are NOT run by executing a Jupyter cell - they must be saved as a `.py` file and launched from the terminal with `streamlit run app.py`, which starts a local web server and opens the dashboard in your browser. The code cells that follow in this notebook are written as if saved into that `app.py` file, purely for teaching purposes.

In [ ]:
# pip install streamlit
# Save the next cell as app.py, then run: streamlit run app.py

## A Minimal Streamlit App
Streamlit's core idea is that each widget or piece of content is added with a single function call, and the library automatically reruns your whole script top-to-bottom whenever the user interacts with the page - so you write plain, linear Python and Streamlit handles the reactivity. `st.title` renders a page heading, and `st.line_chart` takes a DataFrame and instantly draws an interactive line chart from it, with no matplotlib/plotly boilerplate needed. In the example below, we generate seven days of random sales numbers, set `day` as the index (so it becomes the x-axis), and hand it to `st.line_chart`. When this script is run with `streamlit run app.py`, the browser shows a page titled "Sales Dashboard" with a line chart of the random sales trend.

In [ ]:
# app.py
import streamlit as st
import pandas as pd
import numpy as np

st.title('Sales Dashboard')
# Generate 7 days of random sales figures between 20 and 50 for the demo
data = pd.DataFrame({
    'day': range(1, 8),
    'sales': np.random.randint(20, 50, size=7)
})
# set_index('day') makes 'day' the x-axis of the chart instead of a plotted column
st.line_chart(data.set_index('day'))

## Adding Interactivity
A static chart tells one story, but a dashboard becomes genuinely useful when the viewer can adjust what they're looking at without needing to ask a developer to change the code. Streamlit's input widgets - sliders, dropdowns, checkboxes - each return the current value the user has selected, and because Streamlit reruns the whole script on every interaction, using that returned value to filter data is enough to make the whole page update live. Here, `st.slider('Minimum sales', 0, 50, 25)` creates a slider ranging from 0 to 50 with a starting value of 25, and whatever value the viewer drags it to gets stored in `threshold`. `st.write(data[data['sales'] >= threshold])` then filters the sales DataFrame down to rows meeting that threshold and displays it as a table - as the viewer moves the slider, the table updates instantly to show only the qualifying days.

In [ ]:
# app.py (continued)
# Slider returns whatever value the user currently has it set to (default 25)
threshold = st.slider('Minimum sales', 0, 50, 25)
# Filter the data live based on the slider value and display it as a table
st.write(data[data['sales'] >= threshold])

## Uploading Files
A dashboard is even more powerful as a self-serve tool if it doesn't hardcode a fixed dataset but instead lets each user bring their own data to analyze - for example, a manager uploading last month's sales export and immediately seeing a preview or summary, with no coding required on their end. `st.file_uploader` renders a file-picker widget in the browser; while no file has been chosen it returns `None`, and once the user selects a file it returns a file-like object that pandas can read directly. The `if uploaded is not None:` check guards against trying to process "nothing" before a file is chosen. Once a CSV is uploaded, `pd.read_csv(uploaded)` loads it into a DataFrame, and `st.dataframe(df.head())` displays the first few rows in an interactive, scrollable table on the page.

In [ ]:
# app.py (continued)
# Renders a file picker; returns None until the user actually chooses a file
uploaded = st.file_uploader('Upload a CSV', type='csv')
if uploaded is not None:
    # pandas can read the uploaded file object directly, just like a file path
    df = pd.read_csv(uploaded)
    st.dataframe(df.head())